In [1]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)
Line = SimpleNamespace  # img2table Line type stub

# Harness-only conversion helpers: the mined fixture is stored in target-side
# form, while the oracle must receive the semantically equivalent pandas form.
def _to_pandas_fixture(value):
    if isinstance(value, pl.DataFrame):
        return value.to_pandas()
    if isinstance(value, pl.Series):
        return value.to_pandas()
    if isinstance(value, list):
        return [_to_pandas_fixture(item) for item in value]
    if isinstance(value, tuple):
        return tuple(_to_pandas_fixture(item) for item in value)
    if isinstance(value, dict):
        return {key: _to_pandas_fixture(item) for key, item in value.items()}
    if isinstance(value, SimpleNamespace):
        return SimpleNamespace(**{
            key: _to_pandas_fixture(item) for key, item in vars(value).items()
        })
    return value

def _to_polars_fixture(value):
    if isinstance(value, pd.DataFrame):
        return pl.from_pandas(value)
    if isinstance(value, pd.Series):
        return pl.from_pandas(value)
    if isinstance(value, list):
        return [_to_polars_fixture(item) for item in value]
    if isinstance(value, tuple):
        return tuple(_to_polars_fixture(item) for item in value)
    if isinstance(value, dict):
        return {key: _to_polars_fixture(item) for key, item in value.items()}
    if isinstance(value, SimpleNamespace):
        return SimpleNamespace(**{
            key: _to_polars_fixture(item) for key, item in vars(value).items()
        })
    return value


/opt/anaconda3/envs/my_nlp_env/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/envs/my_nlp_env/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


pandas: 3.0.2  polars: 1.39.3


In [2]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- ident_explode_cells ---

# --- ident_from_dicts_cross ---
FIX_IDENT_FROM_DICTS_CROSS_G = None
FIX_IDENT_FROM_DICTS_CROSS_L = [
    SimpleNamespace(dict={"x1": 0.0, "x2": 100.0, "y1": 10.0, "y2": 10.0, "width": 100.0, "height": 1.0}),
    SimpleNamespace(dict={"x1": 0.0, "x2": 100.0, "y1": 50.0, "y2": 50.0, "width": 100.0, "height": 1.0}),
]
FIX_IDENT_FROM_DICTS_CROSS_VERTICAL_LINES = [
    SimpleNamespace(dict={"x1": 0.0, "x2": 0.0, "y1": 10.0, "y2": 50.0, "width": 1.0, "height": 40.0}),
    SimpleNamespace(dict={"x1": 100.0, "x2": 100.0, "y1": 10.0, "y2": 50.0, "width": 1.0, "height": 40.0}),
]

# --- ident_with_columns_conditions ---
FIX_IDENT_WITH_COLUMNS_CONDITIONS_CROSS_H_LINES = pl.DataFrame({"x1":[0],"y1":[15],"x2":[300],"y2":[15],"width":[300],"height":[1],"x1_":[0],"y1_":[45],"x2_":[300],"y2_":[45],"l_contained":[False],"r_contained":[False],"l_corresponds":[False],"r_corresponds":[False]})

print("✅ Fixtures loaded")
OCRDataframe = SimpleNamespace  # mock for testing


✅ Fixtures loaded


In [3]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_ident_explode_cells(df_bbox_delimiters=None, df_cells=None):
    if df_bbox_delimiters is None:
        df_bbox_delimiters = pd.DataFrame({"x1_bbox":[0],"y1_bbox":[0],"x2_bbox":[10],"y2_bbox":[10],"dels":[[(0, 10)]]})
    if df_cells is None:
        df_cells = pd.DataFrame({"x1_bbox":[0],"y1_bbox":[0],"x2_bbox":[10],"y2_bbox":[10]})
    try:
        df_bbox_delimiters = df_bbox_delimiters[df_bbox_delimiters["dels"].notnull()].explode(column="dels").reset_index()
        df_bbox_delimiters[['del1', 'del2']] = pd.DataFrame(df_bbox_delimiters.dels.tolist(), index=df_bbox_delimiters.index)
        df_bbox_delimiters["x1_bbox"] = df_bbox_delimiters["del1"]
        df_bbox_delimiters["x2_bbox"] = df_bbox_delimiters["del2"]
        df_cells = df_bbox_delimiters[["x1_bbox", "y1_bbox", "x2_bbox", "y2_bbox"]]
        df_cells.columns = ["x1", "y1", "x2", "y2"]
        return df_cells.reset_index()
    except (ValueError, TypeError, pl.exceptions.InvalidOperationError):
        return pd.DataFrame(columns=["index", "x1", "y1", "x2", "y2"])
    return None

def before_ident_from_dicts_cross(g, l):
    def get_cells_dataframe(horizontal_lines: List[Line], vertical_lines: List[Line]) -> pd.DataFrame:
        default_df = pd.DataFrame(columns=["x1", "x2", "y1", "y2", 'width', "height"])
        df_h_lines = pd.DataFrame(map(lambda l: l.dict, horizontal_lines)) if horizontal_lines else default_df.copy()
        df_v_lines = pd.DataFrame(map(lambda l: l.dict, vertical_lines)) if vertical_lines else default_df.copy()

        df_h_lines_cp = df_h_lines.copy()
        df_h_lines_cp.columns = ["x1_", "x2_", "y1_", "y2_", 'width_', "height_"]

        cross_h_lines = df_h_lines.merge(df_h_lines_cp, how='cross')
        cross_h_lines = cross_h_lines[cross_h_lines["y1"] < cross_h_lines["y1_"]]

        cross_h_lines["l_corresponds"] = (cross_h_lines["x1"] - cross_h_lines["x1_"] / cross_h_lines["width"]).abs() <= 0.02
        cross_h_lines["r_corresponds"] = (cross_h_lines["x2"] - cross_h_lines["x2_"] / cross_h_lines["width"]).abs() <= 0.02
        cross_h_lines["l_contained"] = (((cross_h_lines["x1"] <= cross_h_lines["x1_"])
                                        & (cross_h_lines["x1_"] <= cross_h_lines["x2"]))
                                        | ((cross_h_lines["x1_"] <= cross_h_lines["x1"])
                                           & (cross_h_lines["x1"] <= cross_h_lines["x2_"])))
        cross_h_lines["r_contained"] = (((cross_h_lines["x1"] <= cross_h_lines["x2_"])
                                         & (cross_h_lines["x2_"] <= cross_h_lines["x2"]))
                                        | ((cross_h_lines["x1_"] <= cross_h_lines["x2"])
                                           & (cross_h_lines["x2"] <= cross_h_lines["x2_"])))

        matching_condition = ((cross_h_lines["l_corresponds"] | cross_h_lines["l_contained"])
                              & (cross_h_lines["r_corresponds"] | cross_h_lines["r_contained"]))
        cross_h_lines = cross_h_lines[matching_condition]

        cross_h_lines["x1_bbox"] = cross_h_lines[["x1", "x1_"]].max(axis=1)
        cross_h_lines["x2_bbox"] = cross_h_lines[["x2", "x2_"]].min(axis=1)
        cross_h_lines["y1_bbox"] = cross_h_lines["y1"]
        cross_h_lines["y2_bbox"] = cross_h_lines["y1_"]
        df_bbox = cross_h_lines[["x1_bbox", "y1_bbox", "x2_bbox", "y2_bbox"]].reset_index()

        df_bbox["h_margin"] = pd.concat([(df_bbox["x2_bbox"] - df_bbox["x1_bbox"]) * 0.05,
                                         pd.Series(5.0, index=range(len(df_bbox)))],
                                        axis=1).max(axis=1).round()
        df_bbox_v = df_bbox.merge(df_v_lines, how='cross')

        horizontal_cond = ((df_bbox_v["x1_bbox"] - df_bbox_v["h_margin"] <= df_bbox_v["x1"])
                           & (df_bbox_v["x2_bbox"] + df_bbox_v["h_margin"] > + df_bbox_v["x1"]))
        df_bbox_v = df_bbox_v[horizontal_cond]

        df_bbox_v["overlapping"] = df_bbox_v[["y2", "y2_bbox"]].min(axis=1) - df_bbox_v[["y1", "y1_bbox"]].max(axis=1)
        df_bbox_v = df_bbox_v[df_bbox_v["overlapping"] / (df_bbox_v["y2_bbox"] - df_bbox_v["y1_bbox"]) >= 0.8]

        df_bbox_delimiters = (df_bbox_v.groupby(['index', "x1_bbox", "x2_bbox", "y1_bbox", "y2_bbox"])
                              .agg(dels=('x1', lambda x: [bound for bound in zip(sorted(x), sorted(x)[1:])] or None))
                              )

        try:
            df_bbox_delimiters = df_bbox_delimiters[df_bbox_delimiters["dels"].notnull()].explode(column="dels").reset_index()
            df_bbox_delimiters[['del1', 'del2']] = pd.DataFrame(df_bbox_delimiters.dels.tolist(),
                                                                index=df_bbox_delimiters.index)
            df_bbox_delimiters["x1_bbox"] = df_bbox_delimiters["del1"]
            df_bbox_delimiters["x2_bbox"] = df_bbox_delimiters["del2"]

            df_cells = df_bbox_delimiters[["x1_bbox", "y1_bbox", "x2_bbox", "y2_bbox"]]
            df_cells.columns = ["x1", "y1", "x2", "y2"]

            return df_cells.reset_index()
        except (ValueError, TypeError, pl.exceptions.InvalidOperationError):
            return pd.DataFrame(columns=["index", "x1", "y1", "x2", "y2"])
    return get_cells_dataframe

def before_ident_with_columns_conditions(cross_h_lines):
    cross_h_lines["l_corresponds"] = (cross_h_lines["x1"] - cross_h_lines["x1_"] / cross_h_lines["width"]).abs() <= 0.02
    cross_h_lines["r_corresponds"] = (cross_h_lines["x2"] - cross_h_lines["x2_"] / cross_h_lines["width"]).abs() <= 0.02
    cross_h_lines["l_contained"] = (((cross_h_lines["x1"] <= cross_h_lines["x1_"])
                                    & (cross_h_lines["x1_"] <= cross_h_lines["x2"]))
                                    | ((cross_h_lines["x1_"] <= cross_h_lines["x1"])
                                       & (cross_h_lines["x1"] <= cross_h_lines["x2_"])))
    cross_h_lines["r_contained"] = (((cross_h_lines["x1"] <= cross_h_lines["x2_"])
                                     & (cross_h_lines["x2_"] <= cross_h_lines["x2"]))
                                    | ((cross_h_lines["x1_"] <= cross_h_lines["x2"])
                                       & (cross_h_lines["x2"] <= cross_h_lines["x2_"])))
    return cross_h_lines

_oracle_ident_with_columns_conditions = before_ident_with_columns_conditions
def before_ident_with_columns_conditions(*args, **kwargs):
    return _oracle_ident_with_columns_conditions(
        *[_to_pandas_fixture(value) for value in args],
        **{key: _to_pandas_fixture(value) for key, value in kwargs.items()},
    )


In [4]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_ident_explode_cells(df_bbox_delimiters=None, df_cells=None):
    if df_bbox_delimiters is None:
        df_bbox_delimiters = pl.DataFrame({"x1_bbox":[0],"y1_bbox":[0],"x2_bbox":[10],"y2_bbox":[10],"dels":[[(0, 10)]]})
    if df_cells is None:
        df_cells = pl.DataFrame({"x1_bbox":[0],"y1_bbox":[0],"x2_bbox":[10],"y2_bbox":[10]})

    try:
        df_bbox_delimiters = (
            df_bbox_delimiters.filter(pl.col("dels").is_not_null())
            .explode("dels")
            .with_columns(
                pl.col("dels").list.get(0).alias("del1"),
                pl.col("dels").list.get(1).alias("del2"),
            )
            .with_columns(
                pl.col("del1").alias("x1_bbox"),
                pl.col("del2").alias("x2_bbox"),
            )
        )
        df_cells = df_bbox_delimiters.select(["x1_bbox", "y1_bbox", "x2_bbox", "y2_bbox"])
        df_cells.columns = ["x1", "y1", "x2", "y2"]
        return df_cells.with_row_count("index")
    except ValueError:
        return pl.DataFrame(schema={"index": pl.Int64, "x1": pl.Null, "y1": pl.Null, "x2": pl.Null, "y2": pl.Null})
    return None

def gen_ident_from_dicts_cross(g, l):
    from typing import List

    def get_cells_dataframe(horizontal_lines: List[Line], vertical_lines: List[Line]) -> pl.DataFrame:
        default_df = pl.DataFrame({"x1": [], "x2": [], "y1": [], "y2": [], "width": [], "height": []})
        df_h_lines = pl.DataFrame([l.dict for l in horizontal_lines]) if horizontal_lines else default_df.clone()
        df_v_lines = pl.DataFrame([l.dict for l in vertical_lines]) if vertical_lines else default_df.clone()

        df_h_lines_cp = df_h_lines.clone()
        df_h_lines_cp.columns = ["x1_", "x2_", "y1_", "y2_", "width_", "height_"]

        cross_h_lines = df_h_lines.join(df_h_lines_cp, how="cross")
        cross_h_lines = cross_h_lines.filter(pl.col("y1") < pl.col("y1_"))

        cross_h_lines = cross_h_lines.with_columns([
            ((pl.col("x1") - pl.col("x1_") / pl.col("width")).abs() <= 0.02).alias("l_corresponds"),
            ((pl.col("x2") - pl.col("x2_") / pl.col("width")).abs() <= 0.02).alias("r_corresponds"),
            (
                ((pl.col("x1") <= pl.col("x1_")) & (pl.col("x1_") <= pl.col("x2")))
                | ((pl.col("x1_") <= pl.col("x1")) & (pl.col("x1") <= pl.col("x2_")))
            ).alias("l_contained"),
            (
                ((pl.col("x1") <= pl.col("x2_")) & (pl.col("x2_") <= pl.col("x2")))
                | ((pl.col("x1_") <= pl.col("x2")) & (pl.col("x2") <= pl.col("x2_")))
            ).alias("r_contained"),
        ])

        matching_condition = ((pl.col("l_corresponds") | pl.col("l_contained")) & (pl.col("r_corresponds") | pl.col("r_contained")))
        cross_h_lines = cross_h_lines.filter(matching_condition)

        cross_h_lines = cross_h_lines.with_columns([
            pl.max_horizontal("x1", "x1_").alias("x1_bbox"),
            pl.min_horizontal("x2", "x2_").alias("x2_bbox"),
            pl.col("y1").alias("y1_bbox"),
            pl.col("y1_").alias("y2_bbox"),
        ])
        df_bbox = cross_h_lines.select(["x1_bbox", "y1_bbox", "x2_bbox", "y2_bbox"]).with_row_index("index")

        df_bbox = df_bbox.with_columns(
            pl.max_horizontal(
                (pl.col("x2_bbox") - pl.col("x1_bbox")) * 0.05,
                pl.lit(5.0),
            ).round().alias("h_margin")
        )

        df_bbox_v = df_bbox.join(df_v_lines, how="cross")

        horizontal_cond = (
            (pl.col("x1_bbox") - pl.col("h_margin") <= pl.col("x1"))
            & (pl.col("x2_bbox") + pl.col("h_margin") > pl.col("x1"))
        )
        df_bbox_v = df_bbox_v.filter(horizontal_cond)

        df_bbox_v = df_bbox_v.with_columns(
            (pl.min_horizontal("y2", "y2_bbox") - pl.max_horizontal("y1", "y1_bbox")).alias("overlapping")
        )
        df_bbox_v = df_bbox_v.filter((pl.col("overlapping") / (pl.col("y2_bbox") - pl.col("y1_bbox"))) >= 0.8)

        if df_bbox_v.is_empty():
            return pl.DataFrame({"index": [], "x1": [], "y1": [], "x2": [], "y2": []})

        df_bbox_delimiters = (
            df_bbox_v.group_by(["index", "x1_bbox", "x2_bbox", "y1_bbox", "y2_bbox"])
            .agg(pl.col("x1").sort().alias("x1_sorted"))
        )

        df_bbox_delimiters = df_bbox_delimiters.with_columns(
            pl.when(pl.col("x1_sorted").list.len() > 1)
            .then(
                pl.col("x1_sorted").map_elements(
                    lambda x: list(zip(x, x[1:])) if x is not None and len(x) > 1 else None,
                    return_dtype=pl.List(pl.Object),
                )
            )
            .otherwise(None)
            .alias("dels")
        )

        df_bbox_delimiters = df_bbox_delimiters.filter(pl.col("dels").is_not_null())

        try:
            if df_bbox_delimiters.is_empty():
                raise ValueError

            df_bbox_delimiters = df_bbox_delimiters.explode("dels")
            df_bbox_delimiters = df_bbox_delimiters.with_columns([
                pl.col("dels").list.get(0).alias("del1"),
                pl.col("dels").list.get(1).alias("del2"),
            ])

            df_bbox_delimiters = df_bbox_delimiters.with_columns([
                pl.col("del1").alias("x1_bbox"),
                pl.col("del2").alias("x2_bbox"),
            ])

            df_cells = df_bbox_delimiters.select(["x1_bbox", "y1_bbox", "x2_bbox", "y2_bbox"])
            df_cells.columns = ["x1", "y1", "x2", "y2"]

            return df_cells.with_row_index("index")
        except ValueError:
            return pl.DataFrame({"index": [], "x1": [], "y1": [], "x2": [], "y2": []})
    return get_cells_dataframe

def gen_ident_with_columns_conditions(cross_h_lines):
    cross_h_lines = cross_h_lines.with_columns(
        [
            ((pl.col("x1") - pl.col("x1_") / pl.col("width")).abs() <= 0.02).alias("l_corresponds"),
            ((pl.col("x2") - pl.col("x2_") / pl.col("width")).abs() <= 0.02).alias("r_corresponds"),
            (
                ((pl.col("x1") <= pl.col("x1_")) & (pl.col("x1_") <= pl.col("x2")))
                | ((pl.col("x1_") <= pl.col("x1")) & (pl.col("x1") <= pl.col("x2_")))
            ).alias("l_contained"),
            (
                ((pl.col("x1") <= pl.col("x2_")) & (pl.col("x2_") <= pl.col("x2")))
                | ((pl.col("x1_") <= pl.col("x2")) & (pl.col("x2") <= pl.col("x2_")))
            ).alias("r_contained"),
        ]
    )
    return cross_h_lines

def _to_pandas_fixture(obj):
    if isinstance(obj, pl.Series):
        return obj.to_pandas()
    if isinstance(obj, pl.DataFrame):
        return obj.to_pandas()
    if isinstance(obj, pl.LazyFrame):
        return obj.collect().to_pandas()
    if isinstance(obj, list):
        return [_to_pandas_fixture(x) for x in obj]
    if isinstance(obj, tuple):
        return tuple(_to_pandas_fixture(x) for x in obj)
    if isinstance(obj, dict):
        return {k: _to_pandas_fixture(v) for k, v in obj.items()}
    if hasattr(obj, "df") and isinstance(getattr(obj, "df"), (pl.DataFrame, pl.LazyFrame)):
        return SimpleNamespace(df=_to_pandas_fixture(obj.df))
    return obj

def _to_polars_fixture(obj):
    if isinstance(obj, pd.Series):
        return pl.Series(obj.name or "series", obj.to_list())
    if isinstance(obj, pd.DataFrame):
        return pl.from_pandas(obj)
    if isinstance(obj, list):
        return [_to_polars_fixture(x) for x in obj]
    if isinstance(obj, tuple):
        return tuple(_to_polars_fixture(x) for x in obj)
    if isinstance(obj, dict):
        return {k: _to_polars_fixture(v) for k, v in obj.items()}
    if hasattr(obj, "df") and isinstance(getattr(obj, "df"), pd.DataFrame):
        return SimpleNamespace(df=_to_polars_fixture(obj.df))
    return obj

def _wrap_before_func(fn):
    def _wrapped(*args, **kwargs):
        _old_self_df = None
        if "self" in globals() and hasattr(self, "df"):
            _old_self_df = self.df
            self.df = _to_pandas_fixture(self.df)
        try:
            return fn(*[_to_pandas_fixture(a) for a in args], **{k: _to_pandas_fixture(v) for k, v in kwargs.items()})
        finally:
            if _old_self_df is not None:
                self.df = _old_self_df
    return _wrapped

def _wrap_gen_func(fn):
    def _wrapped(*args, **kwargs):
        _old_self_df = None
        if "self" in globals() and hasattr(self, "df"):
            _old_self_df = self.df
            self.df = _to_polars_fixture(self.df)
        try:
            return fn(*[_to_polars_fixture(a) for a in args], **{k: _to_polars_fixture(v) for k, v in kwargs.items()})
        finally:
            if _old_self_df is not None:
                self.df = _old_self_df
    return _wrapped

for _name, _fn in list(globals().items()):
    if callable(_fn) and _name.startswith("before_"):
        globals()[_name] = _wrap_before_func(_fn)
    elif callable(_fn) and _name.startswith("gen_"):
        globals()[_name] = _wrap_gen_func(_fn)

# ── Test harness type adapters ─────────────────────────────────────────────
def _to_pandas_fixture(obj):
    if isinstance(obj, pl.Series):
        return obj.to_pandas()
    if isinstance(obj, pl.DataFrame):
        return obj.to_pandas()
    if isinstance(obj, pl.LazyFrame):
        return obj.collect().to_pandas()
    if isinstance(obj, list):
        return [_to_pandas_fixture(x) for x in obj]
    if isinstance(obj, tuple):
        return tuple(_to_pandas_fixture(x) for x in obj)
    if isinstance(obj, dict):
        return {k: _to_pandas_fixture(v) for k, v in obj.items()}
    if hasattr(obj, "df") and isinstance(getattr(obj, "df"), (pl.DataFrame, pl.LazyFrame)):
        return SimpleNamespace(df=_to_pandas_fixture(obj.df))
    return obj

def _to_polars_fixture(obj):
    if isinstance(obj, pd.Series):
        return pl.Series(obj.name or "series", obj.to_list())
    if isinstance(obj, pd.DataFrame):
        return pl.from_pandas(obj)
    if isinstance(obj, list):
        return [_to_polars_fixture(x) for x in obj]
    if isinstance(obj, tuple):
        return tuple(_to_polars_fixture(x) for x in obj)
    if isinstance(obj, dict):
        return {k: _to_polars_fixture(v) for k, v in obj.items()}
    if hasattr(obj, "df") and isinstance(getattr(obj, "df"), pd.DataFrame):
        return SimpleNamespace(df=_to_polars_fixture(obj.df))
    return obj

def _wrap_before_func(fn):
    def _wrapped(*args, **kwargs):
        _old_self_df = None
        if "self" in globals() and hasattr(self, "df"):
            _old_self_df = self.df
            self.df = _to_pandas_fixture(self.df)
        try:
            return fn(*[_to_pandas_fixture(a) for a in args], **{k: _to_pandas_fixture(v) for k, v in kwargs.items()})
        finally:
            if _old_self_df is not None:
                self.df = _old_self_df
    return _wrapped

def _wrap_gen_func(fn):
    def _wrapped(*args, **kwargs):
        _old_self_df = None
        if "self" in globals() and hasattr(self, "df"):
            _old_self_df = self.df
            self.df = _to_polars_fixture(self.df)
        try:
            return fn(*[_to_polars_fixture(a) for a in args], **{k: _to_polars_fixture(v) for k, v in kwargs.items()})
        finally:
            if _old_self_df is not None:
                self.df = _old_self_df
    return _wrapped

for _name, _fn in list(globals().items()):
    if callable(_fn) and _name.startswith("before_"):
        globals()[_name] = _wrap_before_func(_fn)
    elif callable(_fn) and _name.startswith("gen_"):
        globals()[_name] = _wrap_gen_func(_fn)


In [5]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if hasattr(r, "df"):
        r = r.df
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [6]:
# === Tests: ident_with_columns_conditions ===

# L1 smoke – generated
try:
    _r = gen_ident_with_columns_conditions(FIX_IDENT_WITH_COLUMNS_CONDITIONS_CROSS_H_LINES)
    print("✅ L1 smoke gen_ident_with_columns_conditions: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_ident_with_columns_conditions: {type(_e).__name__}: {_e}")

# L1 smoke – before
try:
    _rb = before_ident_with_columns_conditions(FIX_IDENT_WITH_COLUMNS_CONDITIONS_CROSS_H_LINES)
    print("✅ L1 smoke before_ident_with_columns_conditions: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_ident_with_columns_conditions: {type(_e).__name__}: {_e}")

# L2 behavioral equivalence
try:
    _rb = before_ident_with_columns_conditions(FIX_IDENT_WITH_COLUMNS_CONDITIONS_CROSS_H_LINES)
    _rg = gen_ident_with_columns_conditions(FIX_IDENT_WITH_COLUMNS_CONDITIONS_CROSS_H_LINES)
    compare(_rb, _rg, "ident_with_columns_conditions")
except Exception as _e:
    print(f"❌ L2 equivalence ident_with_columns_conditions: setup error — {type(_e).__name__}: {_e}")

# L3 edge - compare the schema-bearing empty result with the oracle.
try:
    _rb = before_ident_with_columns_conditions(FIX_IDENT_WITH_COLUMNS_CONDITIONS_CROSS_H_LINES.head(0))
    _rg = gen_ident_with_columns_conditions(FIX_IDENT_WITH_COLUMNS_CONDITIONS_CROSS_H_LINES.head(0))
    compare(_rb, _rg, "L3 edge ident_with_columns_conditions empty input", check_row_order=True)
except Exception as _e:
    print(f"❌ L3 edge ident_with_columns_conditions: {type(_e).__name__}: {_e}")


✅ L1 smoke gen_ident_with_columns_conditions: OK, type= DataFrame
✅ L1 smoke before_ident_with_columns_conditions: OK
✅ L2 equivalence ident_with_columns_conditions: MATCH
✅ L3 edge ident_with_columns_conditions empty input: MATCH
